## 09 — TrustyAI flood demo (workbench)

Observe (15m, refresh 15s):

```promql
trustyai_meanshift{model="muffin-chihuahua"}
```

(Or pin `request="<your-MeanShift-requestId>"` from §8.3.)

Two steps only:

| # | Flood | Send | Expect |
|---|-------|------|--------|
| 1 | **Negative** | synthetic junk (noise, colors, scribbles) | both curves ↓ |
| 2 | **Positive** | real chihuahuas **+** muffins (50/50) | both curves ↑ toward 1 |

Run **one cell**, wait ~1–2 min, then the next. Keep volumes small (~2 MiB stored per image).

In [ ]:
import json
import urllib.request
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw

EP = "http://muffin-chihuahua-predictor.chihuahua-vs-muffin-jan.svc.cluster.local:9081"
MODEL = "muffin-chihuahua"
D = np.dtype("f4")
RNG = np.random.default_rng(42)
TEST_ROOT = Path("/opt/app-root/src/data/muffin-chihuahua/test")


def to_chw_from_img(img: Image.Image) -> np.ndarray:
    img = img.convert("RGB").resize((224, 224))
    arr = np.asarray(img).astype(D) / D.type(255.0)
    mean = np.array([0.485, 0.456, 0.406], dtype=D)
    std = np.array([0.229, 0.224, 0.225], dtype=D)
    arr = (arr - mean) / std
    return np.transpose(arr, (2, 0, 1))[None, ...].astype(D)


def to_chw(path: Path) -> np.ndarray:
    return to_chw_from_img(Image.open(path))


def infer_chw(chw: np.ndarray, label: str) -> None:
    body = json.dumps(
        {
            "inputs": [
                {
                    "name": "input",
                    "shape": list(chw.shape),
                    "datatype": "FP32",
                    "data": chw.ravel().tolist(),
                }
            ]
        }
    ).encode()
    req = urllib.request.Request(
        f"{EP}/v2/models/{MODEL}/infer",
        data=body,
        headers={"Content-Type": "application/json"},
    )
    try:
        st = urllib.request.urlopen(req, timeout=180).status
        print(label, st)
    except Exception as e:
        print(label, "FAIL", e)


print("EP", EP)
print("test root", TEST_ROOT, "exists", TEST_ROOT.exists())

### 1) Negative flood — junk traffic

Synthetic images far from TRAINING → both MeanShift series should **drop**.

In [ ]:
print("=== NEGATIVE flood (junk) ===")
for i in range(6):
    kind = i % 4
    if kind == 0:
        img = Image.fromarray(RNG.integers(0, 255, (224, 224, 3), dtype=np.uint8), "RGB")
        tag = "noise"
    elif kind == 1:
        color = tuple(int(x) for x in RNG.integers(0, 255, 3))
        img = Image.new("RGB", (224, 224), color)
        tag = f"solid{color}"
    elif kind == 2:
        img = Image.new("RGB", (224, 224), (255, 255, 255))
        d = ImageDraw.Draw(img)
        for _ in range(30):
            xy = [tuple(int(x) for x in RNG.integers(0, 224, 2)) for _ in range(2)]
            d.line(xy, fill=tuple(int(x) for x in RNG.integers(0, 255, 3)), width=3)
        tag = "scribble"
    else:
        img = Image.new("RGB", (224, 224), (0, 0, 0))
        d = ImageDraw.Draw(img)
        d.text((20, 100), f"JUNK {i}", fill=(0, 255, 0))
        tag = "text"
    infer_chw(to_chw_from_img(img), f"neg:{tag}:{i}")
print("done — wait ~1–2 min, watch BOTH curves ↓")

### 2) Positive flood — in-domain traffic

Real **chihuahuas + muffins** (same mix as TRAINING) → both series should **rise** toward 1.

In [ ]:
print("=== POSITIVE flood (chihuahua + muffin) ===")
per_class, repeats = 4, 2
for class_name in ("chihuahua", "muffin"):
    folder = TEST_ROOT / class_name
    paths = [p for p in folder.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}][:per_class]
    print(f"pos {class_name}: {len(paths)} images x{repeats}")
    for r in range(repeats):
        for p in paths:
            infer_chw(to_chw(p), f"pos:{class_name}/{p.name}:r{r}")
print("done — wait ~1–2 min, watch BOTH curves ↑ toward 1")